# Stage 5 · Q1 + Q2

The first two questions of the brief's Section 6:

- **Q1** — basket size, in-basket weighting scheme, average daily turnover
- **Q2** — headline performance table at 50M / 250M / 1B AUM (under the Section 6.3 cost schedule)

**Code**: `q1.py` (hyper-parameter selection), `q2.py` (headline-table assembly).
The shared engine `stage5_portfolio.py` (data loading / metrics / QuantStats) is read-only.
Q3/Q4/Q5 are handled by the teammates' `q3.py` / `q4.py` / `q5.py`.

## Input data (from Steps 2 / 3 / 4)

| Source | File | Used for |
| --- | --- | --- |
| Step 4 XGBoost2 | `xgboost2_predictions_all.parquet` | realised overnight return `r_on_next` + XGB rank score + sample split |
| Step 4 Ridge | `ridge_oos_predictions.parquet` | Ridge out-of-sample score (valid 2019–21 + test 2022–24) |
| Step 2 | `stage2_capacity_eligibility.parquet` | trade-eligible flag + per-AUM participation cap (5%·ADV20) |
| Step 3 | `step3_borrow_tiers.parquet` | daily short borrow cost + tier |

> Honest out-of-sample window = 2019–2024 (both models have OOS scores there). 2010–2018 is the XGBoost in-sample training period and is not used for the headline.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import stage5_portfolio as s5                                                    # shared engine (data/metrics/reporting)
from q1 import assert_config_in_sync, basket_turnover_table, select_hyperparams  # Q1 logic
from q2 import headline_table, ensemble_robustness                              # Q2 logic
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)

cfg = s5.Step5Config()       # fixed Section 6.3 cost schedule + the three AUM levels
panel = s5.load_panel(cfg)   # merge the four sources into one point-in-time panel
print('panel rows:', len(panel), '| dates:', panel.date.min().date(), '->', panel.date.max().date())

## Q1 · basket size / weighting scheme / average turnover

**Basket size**: take a top/bottom `quantile` of the cross-section for the long/short legs.
Together with the ensemble weight `w_ridge`, it is searched **jointly by net Sharpe on the
validation split (2019–2021) only** (the test set is never used for tuning).

In [ ]:
# Joint (w_ridge, quantile) search on validation -- logic lives in q1.py
w_best, q_best, grid = select_hyperparams(panel, cfg)
assert_config_in_sync(w_best, q_best)
cfg.w_ridge, cfg.quantile = w_best, q_best
print(f'selected on validation: w_ridge={w_best}, quantile={q_best}')
grid.pivot(index='w_ridge', columns='quantile', values='valid_net_sharpe').round(2)

> The validation net-Sharpe maximum is at `w_ridge=0.3, quantile=0.02`.
> Intuition: a decile (0.10) dilutes the long/short spread to ~2.7 bps/day < the 4 bps/day
> round-trip cost → net return must be negative; only concentrating the book on the
> highest-conviction extreme scores (~2%) lets gross return clear the cost. This is the
> core trade-off of the strategy.

**Weighting scheme**: equal weight inside each basket, then iterative pro-rata
redistribution under the `5%·ADV20` per-name cap via `water_fill` (Section 6.2).
Gross = 100%·AUM (50% long / 50% short), strictly dollar-neutral.

In [ ]:
# Q1 basket + turnover: call q1.py's basket_turnover_table (single source; identical to `python q1.py`)
# XGBoost-only = headline strategy; OOS 2019-2024 = honest reporting window.
q1 = basket_turnover_table(panel, cfg)
q1.to_csv('Stage_5_portfolio_outputs/stage5_q1_basket_turnover.csv', index=False)
q1

> Turnover falls as AUM rises: 50M is near fully turned over (~1.00), 250M ~0.90, 1B ~0.50.
> This is the capacity effect: at larger AUM the `5%·ADV20` participation cap binds more often,
> so the book cannot always absorb the target gross exposure.
> The realised OOS basket holds a median of 19 names per side (daily range 14–20).

## Q2 · headline performance table (50M / 250M / 1B)

**Model choice**: the headline uses the **XGBoost-only score** — it is the only score with
full 2010–2024 coverage (Ridge exists only for 2019–2024).

- `full_2010_2024`: satisfies the brief's literal "2010–2024" request, but **2010–2018 is
  in-sample** (`contains_insample=True`) → inflated, must be flagged in the report.
- `oos_2019_2024` / `test_2022_2024`: the honest out-of-sample figures.

The Ridge+XGB **ensemble is reported separately as an OOS robustness check** (second table
below), not in the headline.

In [ ]:
# headline (XGBoost-only, full window + OOS) -- logic lives in q2.py
table, store = headline_table(panel, cfg)
table.to_csv('Stage_5_portfolio_outputs/stage5_headline_performance.csv')
display(table[['model', 'contains_insample', 'basket_quantile', 'cost_round_trip_bps',
               'n_days', 'net_ann_return', 'net_ann_vol', 'net_sharpe',
               'gross_sharpe', 'max_drawdown', 'avg_daily_turnover',
               'avg_gross_exposure', 'frac_days_at_cap', 'borrow_sharpe_drag']].round(3))

# ensemble as OOS robustness only (Ridge+XGB vs XGB-only), not the headline table.
robust = ensemble_robustness(panel, cfg)
robust.to_csv('Stage_5_portfolio_outputs/stage5_ensemble_robustness.csv')
print('\nEnsemble robustness (OOS only) -- net Sharpe:')
display(robust['net_sharpe'].unstack('model').round(2))



**How to read it (250M as the reference)**:

- **full_2010_2024**: net Sharpe ≈ 2.45, but `contains_insample=True` → this is the brief's
  literal "2010–2024" number and is **inflated by the in-sample 2010–2018 period**.
- **Out-of-sample decay**: same XGB-only, 250M goes from full-window 2.45 → OOS (2019–24)
  **0.47** → test (2022–24) **−0.04**. This is exactly the *in-sample → OOS degradation*
  the brief asks for.
- Gross Sharpe stays well above net even OOS (250M: 1.54 → 0.47), showing the **4 bps/day
  round-trip cost** is the main drag; the overnight gross alpha must be very concentrated to
  have a chance of covering it.
- The larger the AUM, the more often the participation cap binds (250M ~28% of days, 1B ~79%)
  → realised gross exposure drops from ~0.90 at 250M to ~0.50 at 1B (capacity effect).
- **Ensemble robustness**: at 250M OOS, ensemble 0.42 vs XGB-only 0.47 (close); at 1B,
  ensemble 0.43 > XGB 0.25 (Ridge helps at large AUM).

> The report must state clearly: the `full_2010_2024` headline includes in-sample 2010–2018
> and is not an achievable result; the honest figures are the OOS ones.